# Probing and steering a concept in a language model, step by step

This notebook trains a **linear probe** of a concept (valence: whether a text tells of something
good or bad) in a small open-weights model, and then asks what it really measures and whether
the model uses it. It goes through the questions one should ask of any probe:

1. **A first probe**: what goes into the model, a direction per layer, the layer, the numbers and their controls.
2. **Where to train and where to read**: which tokens, which role, which layer, and the bias of choosing it.
3. **How many examples**: learning curves against their null.
4. **Who writes the texts**: texts the model writes under a system prompt, and how far a probe generalizes.
5. **Surface or concept**: format, confounds, keywords, negation, and how to remove what should not be read.
6. **Does the model use it**: steering against random directions, ablation and patching.
7. **Model-specific traps and null controls**: massive activations, norms, and every control together.
8. **Beyond one direction**: days of the week as a circle, categories, numbers.
9. **Do you need two poles**: a pole against a baseline, and concepts without an opposite.
10. **What probes are good for**: cost, monitoring while generating, side effects of steering.
11. **Is the dataset any good**: an audit and an attack.
12. **Your own probe**.

It uses [concept-probe](https://github.com/mneuronico/concept-probe) and runs on Colab's free GPU
(a T4) in about 15 minutes, plus a few to install the library and download the model. Every number it reports comes with a control: what random directions,
shuffled labels or a baseline give on the same items.

**Before you start**: *Runtime → Change runtime type → T4 GPU*. The default model, Llama 3.2 1B
Instruct, needs a Hugging Face token: accept the license at
[its page](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct), create a token, and add it to
Colab's secrets (the key icon on the left) as `HF_TOKEN`. To skip that, set `MODEL_ID` below to
`"Qwen/Qwen2.5-1.5B-Instruct"`, which needs no token (its numbers will differ).

## 0. Setup

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("concept_probe") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "concept-probe[stats,plots] @ git+https://github.com/mneuronico/concept-probe.git@v0.3.0"], check=True)

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Image, display

import concept_probe as cp
from concept_probe import (Ablate, ConceptProbe, EvalSet, ModelBundle, Patch, Readout, Target, auroc, cohen_d,
                           evaluate_steering, fit_directions, load_example, read_texts)

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"   # or "Qwen/Qwen2.5-1.5B-Instruct": no token needed

try:  # in Colab, the token from the secrets panel
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
if MODEL_ID.startswith("meta-llama/") and not os.environ.get("HF_TOKEN"):
    print("No HF_TOKEN found. Llama needs one (see the top of the notebook), unless you are logged in to Hugging "
          "Face already; or set MODEL_ID to Qwen/Qwen2.5-1.5B-Instruct.")

Loading picks a precision the GPU computes in (float16 on a T4) and says so. Every choice the
library makes for you is printed like that, and saved with the results; results are saved under
`concept_probe_outputs/`.

In [ ]:
bundle = ModelBundle.load(MODEL_ID)
model, tok = bundle.model, bundle.tokenizer
L_ALL = bundle.num_layers
print(f"{MODEL_ID}: {L_ALL} layers")


def show_heatmaps(scores):
    """Show the token heatmaps a Scores object saved."""
    display(HTML(open(os.path.join(scores.out_dir, "heatmaps.html"), encoding="utf-8").read()))

## 1. A first probe

**The data.** 224 minimal pairs: each is one situation told with a good and with a bad outcome,
in many topics and genres. 141 pairs train the probe; 83 evaluate it.

In [ ]:
valence = load_example("valence")
print(valence.description, "\n")
for pos, neg in list(zip(valence.train_pos, valence.train_neg))[:3]:
    print("+", pos, "\n-", neg, "\n")

**What the model sees.** By default a text is read alone: the beginning-of-sequence token and the
text, with no chat template. `preview_input` shows the exact input, and which tokens the probe
reads (the text's own: the first token, which every text has, is left out).

In [ ]:
p = cp.preview_input(tok, valence.train_pos[0])
print(repr(p["input_text"]))
print("the probe reads:", repr(p["span_text"]))
p = cp.preview_input(tok, valence.train_pos[0], role="user", system="You are a helpful assistant.")
print(repr(p["input_text"]))

**Read, fit, evaluate.** A probe is one direction per layer: here, the difference between the
mean representation of the positive texts and that of the negative ones (each text's
representation is the mean over its tokens). The layer is chosen on half of the eval pairs, and
every number comes from the other half, next to its controls: `null_d95` is what random
directions give on the same items, and a refit with shuffled labels says what the procedure
finds with no concept at all.

In [ ]:
probe = ConceptProbe.from_texts(bundle, valence.train_pos, valence.train_neg, valence.eval_pos, valence.eval_neg,
                                paired=True, name="valence")
m = probe.metrics
L = probe.layer
print(f"layer {L} of {L_ALL}: d {m['d']:.2f} {m['d_ci95']}, random directions {m['null_d95']:.2f}, "
      f"AUROC {m['auroc']:.3f}, accuracy {m['accuracy']:.3f}")
print("shuffled labels give |d| up to", round(m["label_shuffle"]["null_abs_d95"], 2))

**On Llama 3.2 1B**: layer 7 of 16, d about 2.4 (random directions 0.4), AUROC 0.95.

In [ ]:
ev = probe.evaluation
layers = np.arange(L_ALL)
plt.figure(figsize=(7, 3.5))
plt.plot(layers, ev.held_out_d_by_layer, "o-", label="held-out half (the reported d)")
plt.plot(layers, ev.selection_d_by_layer, "o-", alpha=0.5, label="selection half (chose the layer)")
plt.fill_between(layers, 0, ev.held_out_null_d95_by_layer, color="grey", alpha=0.3, label="random directions (95th pct)")
plt.axvline(L, color="k", ls=":")
plt.xlabel("layer"); plt.ylabel("Cohen's d"); plt.title("d by layer"); plt.legend(); plt.show()

**New texts.** The score of a text is its projection on the direction; z is that score with the
midpoint between the training poles as 0 and the poles' spread as unit. z = 0 is *not* "without
the concept": it is halfway between the poles. Token by token, the heatmap shows where the
concept is read.

In [ ]:
new = ["The storm passed and nobody was hurt.", "The bakery on the corner closed for good.",
       "My sister finally got the scholarship.", "The meeting is on Thursday at ten."]
for text, z in zip(new, probe.z(read_texts(model, tok, new))):
    print(f"{z:+.2f}  {text}")
show_heatmaps(probe.score_texts(new))

**Four estimators.** The difference of means is one way to get a direction; logistic regression
and LDA are others, and dividing each coordinate by its variance (`mean_diff_var`) is a cheap
version of LDA. Do they find the same direction? Classifying well is not the same as finding the
direction the model uses (section 6 steers with them).

In [ ]:
P, N = read_texts(model, tok, valence.train_pos), read_texts(model, tok, valence.train_neg)
EP, EN = read_texts(model, tok, valence.eval_pos), read_texts(model, tok, valence.eval_neg)
eval_set = EvalSet("eval", EP, EN, paired=True)
estimators = {}
for est in ["mean_diff", "mean_diff_var", "logistic", "lda"]:
    t0 = time.time()
    estimators[est] = ConceptProbe.fit(P, N, eval_sets=eval_set, estimator=est, name=f"valence_{est}", model=bundle,
                                       n_label_shuffles=0, save=False)
    e = estimators[est]
    cos = float(e.directions[L] @ probe.directions[L])
    print(f"{est:14s} layer {e.layer:2d}  d {e.metrics['d']:.2f}  AUROC {e.metrics['auroc']:.3f}  "
          f"cosine with the difference of means at layer {L}: {cos:+.2f}  ({time.time() - t0:.0f} s)")

**On Llama 3.2 1B**: all four choose layer 7. The normalized ones separate the eval pairs better
(d about 3) while pointing elsewhere: logistic regression and LDA have a cosine of 0.5-0.6 with
the difference of means.

## 2. Where to train and where to read

**Which tokens.** The same texts read six ways in one pass: the mean, the last token or the last
12 tokens of the text, and the same over everything from the start of the input to the text's
end (`through_text_*`, which includes the beginning-of-sequence token). A direction trained with
each, read with each, at the probe's layer.

In [ ]:
READOUTS = ["text_mean", "text_last", "text_last_k", "through_text_mean", "through_text_last", "through_text_last_k"]
R = {k: read_texts(model, tok, texts, READOUTS)
     for k, texts in [("P", valence.train_pos), ("N", valence.train_neg), ("EP", valence.eval_pos), ("EN", valence.eval_neg)]}
matrix = np.zeros((len(READOUTS), len(READOUTS)))
for i, train in enumerate(READOUTS):
    v = fit_directions("mean_diff", R["P"][train].data, R["N"][train].data)[L]
    for j, read in enumerate(READOUTS):
        matrix[i, j] = cohen_d(R["EP"][read].data[:, L] @ v, R["EN"][read].data[:, L] @ v)
plt.figure(figsize=(6, 5))
plt.imshow(matrix, cmap="viridis")
plt.xticks(range(6), READOUTS, rotation=45, ha="right"); plt.yticks(range(6), READOUTS)
for i in range(6):
    for j in range(6):
        plt.text(j, i, f"{matrix[i, j]:.1f}", ha="center", va="center", color="w")
plt.xlabel("read with"); plt.ylabel("trained with"); plt.title(f"eval d at layer {L}"); plt.colorbar(); plt.show()

Where to train and where to read are two decisions. The mean over the text is usually the most
robust; the last token reads the end of a sentence (often punctuation); `through_text_mean`
averages in the beginning-of-sequence token, whose activations are huge, so its weight shrinks
with the text's length and the probe drags length along.

**Token by token.** A probe trained on means can score each token. To score a text from its
tokens, the mean or the maximum? On texts whose valence is everywhere, and on *needles*: neutral
paragraphs with a single sentence of valence.

In [ ]:
needles = load_example("valence_needles")
sets = {"valence everywhere": (valence.eval_pos, valence.eval_neg), "one sentence of valence": (needles.pos, needles.neg)}
for name, (pos, neg) in sets.items():
    s_pos, s_neg = probe.score_texts(pos, save=False), probe.score_texts(neg, save=False)
    for how in ["mean", "max", "p90"]:
        print(f"{name:25s} {how:4s} AUROC {auroc(s_pos.per_text(how), s_neg.per_text(how)):.3f}")
show_heatmaps(probe.score_texts([needles.pos[0], needles.neg[0]]))

**On Llama 3.2 1B**: the mean wins on both sets (AUROC 0.95 and 0.83), even on the needles: one
sentence of valence moves the mean enough, and single tokens are noisy.

**The role and the template.** The same eval texts read as a user turn, as the model's own reply,
and after a system prompt. Pooling over the text's own tokens keeps the template out, so the
scores barely move; a readout that includes the template tokens (`through_text_mean`) moves them,
and a z measured on one reading is not valid for another.

In [ ]:
readings = {"raw text": {}, "user turn": {"role": "user"},
            "user turn after a system prompt": {"role": "user", "system": "You are a helpful assistant."},
            "assistant reply": {"role": "assistant", "prompt": "Tell me something that happened."}}
unit = probe.calibrations[0].scale[L]          # the probe's within-pole SD at its layer
v = probe.directions[L]
for readout in ["text_mean", "through_text_mean"]:
    print(f"read with {readout}:")
    base = None
    for name, kw in readings.items():
        sp = read_texts(model, tok, valence.eval_pos, readout, layers=[L], **kw).data[:, 0] @ v
        sn = read_texts(model, tok, valence.eval_neg, readout, layers=[L], **kw).data[:, 0] @ v
        mean = float(np.r_[sp, sn].mean())
        base = mean if base is None else base
        print(f"  {name:32s} shift of the mean score {(mean - base) / unit:+6.2f} SD   d {cohen_d(sp, sn):.2f}")

**On Llama 3.2 1B**: with the text's own tokens the role moves the scores by about 0.1 SD; with
the template tokens in, by 12-14 SD, and d falls from 2.4 to about 1.

Why the first token is left out: its activations are orders of magnitude larger than the rest
(an "attention sink"). Mean norm of the residual stream by position, at a few layers:

In [ ]:
T = read_texts(model, tok, load_example("neutral").texts[:40], "through_text_tokens")
norms = np.linalg.norm(T.data, axis=-1)            # (tokens, layers)
plt.figure(figsize=(7, 3))
for layer in [1, L_ALL // 2, L_ALL - 1]:
    by_pos = [norms[T.positions == k, layer].mean() for k in range(12)]
    plt.semilogy(range(12), by_pos, "o-", label=f"layer {layer}")
plt.xlabel("position in the input"); plt.ylabel("mean norm"); plt.legend(); plt.show()

**Choosing the layer is a multiple comparison.** A concept that does not exist: two random groups
of neutral texts, drawn 50 times. For each draw, a probe is fitted on part of the texts and its d
measured by layer on the rest. Choosing the best layer and reporting its d on the same items gives
a d above 0 with no concept at all; choosing on half of the eval items and measuring on the other
half does not. `ConceptProbe.fit` does the second by default, and adds a permutation p that
corrects the largest d for the search over layers.

In [ ]:
neutral = load_example("neutral")
NT = read_texts(model, tok, neutral.texts)
rng = np.random.default_rng(0)
same_items, held_out = [], []
for _ in range(50):
    o = rng.permutation(len(neutral.texts))
    A, B = NT.data[o[:75]], NT.data[o[75:]]
    dirs = fit_directions("mean_diff", A[:40], B[:40])
    d_sel = [cohen_d(A[40:57, l] @ dirs[l], B[40:57, l] @ dirs[l]) for l in range(L_ALL)]
    d_rep = [cohen_d(A[57:, l] @ dirs[l], B[57:, l] @ dirs[l]) for l in range(L_ALL)]
    same_items.append(max(d_sel))
    held_out.append(d_rep[int(np.argmax(d_sel))])
print(f"no concept, best layer measured on the items that chose it: mean d {np.mean(same_items):+.2f}")
print(f"no concept, best layer measured on held-out items:          mean d {np.mean(held_out):+.2f}")
A, B = NT.data[o[:75]], NT.data[o[75:]]       # one draw, through fit: held-out d and the permutation p
null = ConceptProbe.fit(A[:40], B[:40], eval_sets=EvalSet("eval", A[40:], B[40:]), n_label_shuffles=0,
                        identity=bundle.identity, save=False)
print(f"one draw through ConceptProbe.fit: d {null.metrics['d']:+.2f} (random directions {null.metrics['null_d95']:.2f}), "
      f"permutation p of the largest d {null.evaluation.max_d_permutation_p:.2f}")

**On Llama 3.2 1B**: d +0.30 on the items that chose the layer, 0.00 held out.

## 3. How many examples

Probes fitted on 2 to 128 training pairs, ten draws per size, each measured on the same eval
pairs. The control: the same draws split into two random groups (what a probe of that size
finds with no concept). With few pairs the direction carries the topics of those few texts.

In [ ]:
curve = cp.learning_curve(P.data, N.data, [eval_set], sizes=[2, 4, 8, 16, 32, 64, 128], n_repeats=10, n_null=10,
                          paired=True, layer=L)
sizes = curve["sizes"]
probe_d = [curve["summary"]["probe"][n]["d"] for n in sizes]
probe_sd = [curve["summary"]["probe"][n]["d_sd"] for n in sizes]
null_d95 = [curve["summary"]["null"][n]["d_p95"] for n in sizes]
cosine = [curve["summary"]["probe"][n]["cos_full"] for n in sizes]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].errorbar(sizes, probe_d, probe_sd, marker="o", label="probe")
ax[0].plot(sizes, null_d95, "s--", color="grey", label="random groups (95th pct)")
ax[0].set_xscale("log", base=2); ax[0].set_xlabel("training pairs"); ax[0].set_ylabel("eval d"); ax[0].legend()
ax[1].plot(sizes, cosine, "o-"); ax[1].set_xscale("log", base=2)
ax[1].set_xlabel("training pairs"); ax[1].set_ylabel("cosine with the 141-pair direction")
plt.tight_layout(); plt.show()

## 4. Who writes the texts

**Texts the model writes.** Some concepts are easier to ask for than to write: here, being
interested or bored, as two system prompts. The model answers questions under each, and the
probe is fitted on its answers. Read *with* the system prompt they were written under, the
answers separate easily, partly because the system prompt itself pushes the score; read without
it, d drops. And texts written by hand, read alone, are a third test.

In [ ]:
bored = load_example("bored_vs_interested")
print(bored.extra["pos_system"], "\n")
interest = ConceptProbe.from_system_prompts(
    bundle, bored.extra["pos_system"], bored.extra["neg_system"], bored.extra["train_questions"],
    bored.extra["eval_questions"], eval_texts=(bored.pos, bored.neg), name="interested",
    pos_label=bored.pos_label, neg_label=bored.neg_label)
for name, r in interest.evaluation.sets.items():
    print(f"{name:25s} d {r.d:.2f}  (random directions {r.null_d95:.2f})")

**On Llama 3.2 1B**: d 7.5 read with the system prompt, 2.4 without it, 0.9 on texts written by
hand.

**How far does the valence probe generalize?** The eval pairs, the same situations paraphrased,
genres the training set lacks (headlines, verse), Spanish, and attacks written to fool a probe.
Next to it, a bag-of-words classifier trained on the same training texts. The pairs were written
so that their words barely tell the poles apart (section 11 checks it), so it stays near chance:
what the probe reads is not the words.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

bow = CountVectorizer(ngram_range=(1, 2), min_df=1).fit(valence.train_pos + valence.train_neg)
clf = LogisticRegression(max_iter=2000).fit(bow.transform(valence.train_pos + valence.train_neg),
                                            [1] * len(valence.train_pos) + [0] * len(valence.train_neg))
attacks = load_example("valence_attacks")
ladder = {"eval pairs": (valence.eval_pos, valence.eval_neg)}
for name in ["valence_paraphrase", "valence_genre", "valence_spanish"]:
    ex = load_example(name)
    ladder[name.split("_")[1]] = (ex.pos, ex.neg)
ladder["attacks"] = (attacks.column("text", kind="concept", label="pos"), attacks.column("text", kind="concept", label="neg"))
print(f"{'set':12s} {'probe AUROC':>12s} {'bag of words':>13s}")
for name, (pos, neg) in ladder.items():
    a_probe = auroc(probe.scores(read_texts(model, tok, pos)), probe.scores(read_texts(model, tok, neg)))
    a_bow = auroc(clf.decision_function(bow.transform(pos)), clf.decision_function(bow.transform(neg)))
    print(f"{name:12s} {a_probe:12.3f} {a_bow:13.3f}")

## 5. Surface or concept

**A probe that reads format.** True and false statements where every true one ends with "." and
every false one with "!". The probe separates them perfectly on the same design, and falls apart
when the marks are swapped: it read the punctuation. The heatmap shows the score sitting on the
last token.

In [ ]:
truth = load_example("truth")
bang = lambda texts: [t[:-1] + "!" for t in texts]  # noqa: E731
fmt = ConceptProbe.from_texts(bundle, truth.train_pos, bang(truth.train_neg), truth.eval_pos, bang(truth.eval_neg),
                              paired=True, name="truth_with_marks", n_label_shuffles=0, save=False)
print(f"same design: d {fmt.metrics['d']:.2f} at layer {fmt.layer}")
for name, (pos, neg) in {"no marks (all '.')": (truth.eval_pos, truth.eval_neg),
                         "marks swapped": (bang(truth.eval_pos), truth.eval_neg)}.items():
    d = cohen_d(fmt.scores(read_texts(model, tok, pos)), fmt.scores(read_texts(model, tok, neg)))
    print(f"{name:20s} d {d:+.2f}")
show_heatmaps(fmt.score_texts([truth.eval_pos[0], bang(truth.eval_neg)[0]]))

**On Llama 3.2 1B**: d 4.5 on the same design, at layer 2; about 0 with no marks, and -4.5 with
the marks swapped.

**Five confounds.** Valence pairs where, in training, every positive text has a confound (a
neutral continuation that makes it longer, the past tense, the first person, Spanish, a final "!")
and every negative text lacks it. On eval pairs the two are decoupled: each cell of
valence × confound. A confounded probe separates the confound's levels as much as, or more
than, valence. The clean probe of section 1 should not.

In [ ]:
conf = load_example("valence_confounds")
print(f"{'confound':12s} {'valence d (level 0 / 1)':>24s} {'confound d':>11s}   {'clean probe: confound d':>24s}")
for c in ["length", "tense", "person", "language", "exclamation"]:
    get = lambda key, split: read_texts(model, tok, conf.column(key, confound=c, split=split)).data  # noqa: E731
    v = fit_directions("mean_diff", get("pos_1", "train"), get("neg_0", "train"))[L]
    cells = {k: get(k, "eval")[:, L] for k in ["pos_0", "neg_0", "pos_1", "neg_1"]}
    val_d = [cohen_d(cells["pos_0"] @ v, cells["neg_0"] @ v), cohen_d(cells["pos_1"] @ v, cells["neg_1"] @ v)]
    conf_d = cohen_d(np.r_[cells["pos_1"], cells["neg_1"]] @ v, np.r_[cells["pos_0"], cells["neg_0"]] @ v)
    w = probe.directions[L]
    clean = cohen_d(np.r_[cells["pos_1"], cells["neg_1"]] @ w, np.r_[cells["pos_0"], cells["neg_0"]] @ w)
    print(f"{c:12s} {val_d[0]:11.2f} / {val_d[1]:5.2f} {conf_d:11.2f}   {clean:24.2f}")

**On Llama 3.2 1B**: language and length take over the confounded probe (confound d 8.7 and 3.8,
valence d 0.2-0.8); the exclamation mark does not get in; the clean probe reads none of the five.

**Keywords and negation.** Valence sentences and their negation ("I'm happy" / "I'm not happy"),
neutral sentences with a valence word inside a name ("The Happy Valley bus"), and pairs where the
keywords are replaced by rarer synonyms.

In [ ]:
negated = load_example("valence_negated")
zs = lambda texts: probe.z(read_texts(model, tok, texts))  # noqa: E731
za, zn = zs(negated.column("affirmative")), zs(negated.column("negated"))
sign = np.array([1 if s == "+" else -1 for s in negated.column("sign")])
print(f"negating a valence sentence moves z by {np.mean((zn - za) * -sign):+.2f} toward the other pole "
      f"(flips its side in {np.mean(np.sign(zn) == -sign):.0%} of them)")
names = load_example("valence_in_names")
shift = zs(names.column("with_word")) - zs(names.column("without_word"))
sign = np.array([1 if s == "+" else -1 for s in names.column("sign")])
print(f"a valence word inside a name moves z by {np.mean(shift * sign):+.2f} toward its valence")
syn = load_example("valence_synonyms")
print(f"pairs with rarer synonyms: d {cohen_d(probe.scores(read_texts(model, tok, syn.pos)), probe.scores(read_texts(model, tok, syn.neg))):.2f}")
nn = load_example("neutral_negation")
nshift = zs(nn.column("negated")) - zs(nn.column("affirmative"))
print(f"a negation in a neutral sentence moves z by {nshift.mean():+.2f}")

**On Llama 3.2 1B**: negating a valence sentence moves it 2.6 SD toward the other pole, but a
"not" in a neutral sentence also moves it 1.2 SD toward negative: the probe follows negation
itself.

**Erasing what should not be read.** The probe reads "not" as negative even in neutral sentences.
Fit the negation direction on neutral pairs (train half), erase it from the representations
before fitting (LEACE), and see how much the negation still moves the probe on the other half,
and what that costs in d.

In [ ]:
NNA = read_texts(model, tok, nn.column("affirmative", split="train")).data
NNN = read_texts(model, tok, nn.column("negated", split="train")).data
eraser = cp.Eraser.fit(np.r_[NNA, NNN], [0] * len(NNA) + [1] * len(NNN))
clean_probe = ConceptProbe.fit(P, N, eval_sets=eval_set, layer=L, erase=eraser, name="valence_without_negation",
                               model=bundle, n_label_shuffles=0, save=False)
for name, pr in [("valence probe", probe), ("negation erased", clean_probe)]:
    ea = pr.z(read_texts(model, tok, nn.column("affirmative", split="eval")))
    en = pr.z(read_texts(model, tok, nn.column("negated", split="eval")))
    print(f"{name:16s} d {pr.metrics['d']:.2f}   a negation moves z by {np.mean(en - ea):+.2f}")

**On Llama 3.2 1B**: with the negation erased, a "not" moves the probe by -0.05 instead of -1.13,
and d stays at 2.4.

**Does the probe read the setting?** Positive stories in hospitals and negative ones at parties
would teach a probe "medical = good". Neutral texts of both settings tell whether this probe
reads the setting: `confound_check` compares its d between them with what random directions give.

In [ ]:
tv = load_example("topic_valence")
checks = cp.confound_check(probe.directions, L, {
    "setting (medical vs party)": (read_texts(model, tok, tv.column("text", setting="medical", valence="neutral")).data,
                                   read_texts(model, tok, tv.column("text", setting="party", valence="neutral")).data),
    "negation (neutral)": (read_texts(model, tok, nn.column("negated")).data,
                           read_texts(model, tok, nn.column("affirmative")).data)})
for name, c in checks.items():
    print(f"{name:28s} probe d {c['probe_d']:+.2f} (random {c['null_d95']:.2f}), reads it: {c['reads_confound']}, "
          f"cosine with the confound's direction {c['cosine_at_layer']:+.2f}")

**On Llama 3.2 1B**: neutral hospital texts read more negative than neutral party texts (d -0.97,
at the edge of the 0.95 random directions give), and so does negation.

## 6. Does the model use what the probe reads?

**Steering, against random directions.** Add the probe's direction to the residual stream while
the model writes, at doses in residual-stream norms (0.1 adds a tenth of a typical token's norm),
and the same doses along ten random directions of the same length. The effect is measured by a
judge that is not the probe: sentence embeddings (MiniLM) with a logistic regression trained on
the training pairs. Perplexity and repetition say where the text stops being coherent.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cpu")
embed = lambda texts: embedder.encode(list(texts), normalize_embeddings=True, show_progress_bar=False)  # noqa: E731
judge = LogisticRegression(max_iter=2000).fit(embed(valence.train_pos + valence.train_neg),
                                              [1] * len(valence.train_pos) + [0] * len(valence.train_neg))
positivity = lambda texts: judge.predict_proba(embed(texts))[:, 1]  # noqa: E731
print("judge accuracy on the eval pairs:",
      np.mean(np.r_[positivity(valence.eval_pos) > 0.5, positivity(valence.eval_neg) < 0.5]))

PROMPTS = ["Write three sentences about a man arriving at a train station.",
           "Continue the story in two sentences: When Ana opened the door, she saw",
           "Write a short note to a coworker about tomorrow's meeting.", "Describe a small town at the end of the day.",
           "Write two sentences about a dog in a park.", "Tell a very short story about a student getting her exam results.",
           "Describe a family dinner in three sentences.", "Write a short message to a neighbor about the garden."]

In [ ]:
steer = evaluate_steering(probe, PROMPTS, doses=[-0.25, -0.1, 0.1, 0.25], measures={"positivity": positivity})
for dose in steer.doses:
    row = steer.row(dose)
    e = row["effects"]["positivity"]
    print(f"dose {dose:+.2f}: positivity {e['mean']:.2f} (change {e['change']:+.2f}, random directions up to "
          f"{e['null_abs_change95'] if e.get('null_abs_change95') is not None else float('nan'):.2f}), "
          f"coherent: {row['coherence']['coherent']}")
print("largest coherent dose on each side:", steer.largest_coherent_dose)
steer.plot("steering.png")
display(Image("steering.png"))

**On Llama 3.2 1B**: pushing negative lowers the judge's positivity by 0.22 at -0.25 (random
directions: up to 0.04); pushing positive barely moves it (+0.05, random up to 0.06), since
without steering the model already writes positive texts (0.74). Every dose stays coherent.

In [ ]:
for dose in [-0.25, 0.0, 0.25]:
    item = next(it for it in steer.scores.items if it.info["dose"] == dose and it.info["control"] is None)
    print(f"dose {dose:+.2f}: {item.text!r}\n")

**Decodable is not the same as used: ablation.** A task that needs valence: "Is the following text
positive or negative?", measured as the log-odds of "Positive" over "Negative". Remove the probe's
direction from the residual stream at every layer, and compare with removing a random direction.
If the model still does the task, the information is also elsewhere, or the model does not use
this direction for it.

In [ ]:
INSTR = "Is the following text positive or negative? Answer with one word.\n\nText: "
task = [INSTR + t for t in valence.eval_pos + valence.eval_neg]
truth_label = np.r_[np.ones(len(valence.eval_pos)), np.zeros(len(valence.eval_neg))]
rand = np.random.default_rng(0).standard_normal(probe.directions.shape).astype(np.float32)
rand /= np.linalg.norm(rand, axis=1, keepdims=True)
every = lambda d: {l: d[l] for l in range(len(d))}  # noqa: E731
conditions = {"intact": None, "valence direction removed": Ablate(every(probe.directions)),
              "random direction removed": Ablate(every(rand))}
for name, intervention in conditions.items():
    lo = cp.answer_logprobs(model, tok, task, ["Positive", "Negative"], intervention=intervention, save=False).log_odds()
    print(f"{name:28s} accuracy {np.mean((lo > 0) == truth_label):.3f}   mean |log-odds| {np.abs(lo).mean():.2f}")

**On Llama 3.2 1B**: removing the direction at every layer drops the task from 0.92 to 0.54,
chance; removing a random direction does nothing. Here the model does use this direction.

**Activation patching.** Copy the residual stream of a positive prompt's last token into the run
of its negative pair, one layer at a time, and see how far the answer moves toward the positive
one (1 = all the way). Where it starts moving is where the information becomes causal for the
answer; compare it with where the probe reads well.

In [ ]:
n_pairs = 24
pos_prompts, neg_prompts = task[:n_pairs], task[len(valence.eval_pos):len(valence.eval_pos) + n_pairs]
odds = lambda prompts, i=None: cp.answer_logprobs(model, tok, prompts, ["Positive", "Negative"],  # noqa: E731
                                                  intervention=i, save=False).log_odds()
base_pos, base_neg = odds(pos_prompts), odds(neg_prompts)
last = read_texts(model, tok, pos_prompts, Readout("input", "last"), add_generation_prompt=True).data
moved = []
for layer in range(L_ALL):
    out = odds(neg_prompts, Patch(layer, [-1], last[:, layer][:, None]))
    moved.append(np.median((out - base_neg) / (base_pos - base_neg + 1e-9)))
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(L_ALL), moved, "o-", label="patching the last token (fraction moved)")
ax2 = ax.twinx()
ax2.plot(layers, ev.main.d_by_layer, "s--", color="tab:orange", label="probe d")
ax.set_xlabel("layer"); ax.set_ylabel("fraction moved"); ax2.set_ylabel("probe d")
fig.legend(loc="upper left"); plt.show()
print("first layer where patching moves the answer more than halfway:",
      next((l for l, f in enumerate(moved) if f > 0.5), None))

**On Llama 3.2 1B**: patching the last token moves the answer more than halfway from layer 9 on;
the probe reads best at layer 7. Where a concept is most readable and where it decides the answer
need not be the same layer.

## 7. Model-specific traps and null controls

**Massive activations.** In some models a handful of coordinates carry most of the variance; a
raw difference of means is then dominated by them. Share of the variance in the top 1% of
coordinates, by layer, and d by layer for the raw difference of means and the variance-normalized
one.

In [ ]:
X = np.r_[P.data, N.data]
var = X.var(axis=0)                                   # (layers, dim)
top = max(1, var.shape[1] // 100)
share = np.sort(var, axis=1)[:, -top:].sum(axis=1) / var.sum(axis=1)
d_by = {}
for est in ["mean_diff", "mean_diff_var"]:
    dirs = fit_directions(est, P.data, N.data)
    d_by[est] = [cohen_d(EP.data[:, l] @ dirs[l], EN.data[:, l] @ dirs[l]) for l in range(L_ALL)]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(layers, share, "o-"); ax[0].set_xlabel("layer"); ax[0].set_ylabel("variance in the top 1% of coordinates")
for est, d in d_by.items():
    ax[1].plot(layers, d, "o-", label=est)
ax[1].set_xlabel("layer"); ax[1].set_ylabel("eval d"); ax[1].legend(); plt.tight_layout(); plt.show()

**Norms grow with depth.** A raw score depends on the layer's scale, so raw scores are not compared
across layers or probes: use d or z. Steering doses are given in residual norms for the same reason.

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(layers, P.norms.mean(axis=0), "o-")
plt.xlabel("layer"); plt.ylabel("mean residual norm"); plt.show()

**Every null control together.** The probe's d by layer on all eval pairs, with what random
directions give, what a probe with shuffled labels gives, and what the "concept" of two random
halves of neutral texts gives (section 2).

In [ ]:
null_probe = ConceptProbe.fit(A[:50], B[:50], eval_sets=EvalSet("eval", A[50:], B[50:]), n_label_shuffles=0,
                              identity=bundle.identity, save=False)
plt.figure(figsize=(7, 3.5))
plt.plot(layers, ev.main.d_by_layer, "o-", label="valence probe")
plt.fill_between(layers, 0, ev.main.null_d95_by_layer, color="grey", alpha=0.3, label="random directions (95th pct)")
plt.axhline(m["label_shuffle"]["null_abs_d95"], color="tab:red", ls="--", label="shuffled labels (95th pct |d|)")
plt.plot(layers, np.abs(null_probe.evaluation.main.d_by_layer), "o-", color="tab:green", alpha=0.6,
         label="two halves of neutral texts")
plt.xlabel("layer"); plt.ylabel("eval d"); plt.legend(); plt.show()

## 8. Beyond one direction

**Days of the week.** Sentences ending with a day's name, the same carrier sentences for every
day. Is "day of the week" a direction, or a circle? `target_shape` asks whether the day centroids
have the shape of a circle in order, against the same with the days shuffled. A linear decoder
reads seven classes in 2048 dimensions whatever their arrangement, so its shuffled control is as
high as the real one; decoding *through the shape* (the plane of the centroids) only works if the
days sit in their order.

In [ ]:
dm = load_example("days_months")
day_texts, day_index = dm.column("text", kind="day"), np.array(dm.column("index", kind="day"))
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
D = read_texts(model, tok, day_texts, "text_last")            # the day's name is the last word
target = Target.cycle(day_index, list(range(7)))
shape = cp.target_shape(D, target, n_shuffles=100)
best = shape.best_layer
print(f"best layer {best}: shape R² {shape.shape_r2[best]:.2f} (shuffled days {shape.null_shape_r2_95[best]:.2f})")
for decoder in ["linear", "shape"]:
    r = cp.read_target(D, target, decoder=decoder, layers=[best], n_shuffles=20)
    print(f"{decoder:6s} decoder: accuracy {r.score_by_layer[best]:.2f} (shuffled {r.null_score95_by_layer[best]:.2f}, "
          f"chance {r.chance:.2f})")
c = shape.coords[best]
plt.figure(figsize=(4, 4))
plt.plot(*np.r_[c, c[:1]].T, "-", color="grey")
for k, name in enumerate(DAYS):
    plt.scatter(*c[k]); plt.annotate(name, c[k])
plt.title(f"day centroids, layer {best} (top 2 directions)"); plt.axis("equal"); plt.show()

**On Llama 3.2 1B**: a circle in order at layer 11 (shape R² 0.96, shuffled 0.54). The linear
decoder reads 0.94 of the days, and 1.00 when they are shuffled; the shape decoder 0.71, and 0.46
shuffled.

**Categories and numbers.** Four emotions are not a line: read them as classes (accuracy against
chance). Years are a number: a linear map from the representation to the value (R²).

In [ ]:
emo = load_example("emotions")
E = read_texts(model, tok, emo.texts)
r = cp.read_target(E, Target.categories(emo.column("emotion")), decoder="centroid", n_shuffles=10)
print(f"emotions: accuracy {r.best_score:.2f} at layer {r.best_layer} (chance {r.chance:.2f})")
ordinal = load_example("ordinal")
Y = read_texts(model, tok, ordinal.column("text", variable="year"), "text_mean")
r = cp.read_target(Y, Target.number(ordinal.column("value", variable="year")), decoder="linear", n_shuffles=10)
print(f"years: R² {r.best_score:.2f} at layer {r.best_layer} (shuffled {r.null_score95_by_layer[r.best_layer]:.2f})")

**On Llama 3.2 1B**: emotions 0.85 (chance 0.25); years R² 0.92 already at layer 0: the digits
themselves carry the value, so a number probe can be lexical.

## 9. Do you always need two poles?

**One pole against a baseline.** Positive against negative, or positive against neutral texts?
The second mixes valence with "emotional at all" (what positive and negative share against
neutral). And the neutral texts do not fall at z = 0 of the two-pole probe, which is only the
midpoint between its poles. In the poles' scale, +1 is the positive mean and -1 the negative one.

In [ ]:
NTR, NEV = NT.data[:100], NT.data[100:]
pos_vs_neutral = ConceptProbe.fit(P, NTR, eval_sets=EvalSet("eval", EP.data[:50], NEV), layer=L, zero="baseline",
                                  name="positive_vs_neutral", model=bundle, n_label_shuffles=0, save=False)
neg_vs_neutral = fit_directions("mean_diff", N.data, NTR)[L]
print(f"cosine(positive - negative, positive - neutral) {probe.directions[L] @ pos_vs_neutral.directions[L]:+.2f}")
print(f"cosine(positive - neutral, negative - neutral)  {pos_vs_neutral.directions[L] @ neg_vs_neutral:+.2f}")
print(f"neutral texts on the two-pole probe, in the poles' scale: {probe.z(NT, unit='poles').mean():+.2f}")

**On Llama 3.2 1B**: cosine 0.13 between positive - negative and positive - neutral, and 0.89
between positive - neutral and negative - neutral: most of what sets either pole apart from
neutral texts is shared. Neutral texts sit at +0.60, closer to the positive pole than to the
midpoint.

**Concepts without an opposite.** "Talks about food" has no natural opposite: food against
neutral texts, or food against another topic (cars)? The two directions are not the same; the
baseline is part of the definition.

In [ ]:
topics = load_example("topics")
FOOD = read_texts(model, tok, topics.column("text", set="food")).data
CARS = read_texts(model, tok, topics.column("text", set="cars")).data
v_neutral = fit_directions("mean_diff", FOOD, NT.data)[L]
v_cars = fit_directions("mean_diff", FOOD, CARS)[L]
print(f"cosine(food - neutral, food - cars) at layer {L}: {v_neutral @ v_cars:+.2f}")

**On Llama 3.2 1B**: cosine 0.74: related, not the same direction.

## 10. What probes are good for

**Cost.** Reading a text with a probe is one forward pass and a dot product; asking the model is a
forward pass over the question and the text; generating an answer is one pass per token. In
batches on a GPU the gap per text is small (on Llama 3.2 1B: 8, 16 and 14 ms); the probe's real
advantage is that it reads activations the model computes anyway, as the next cell shows.

In [ ]:
texts = valence.eval_pos + valence.eval_neg
t0 = time.time(); probe.scores(read_texts(model, tok, texts, cache=False)); t_probe = (time.time() - t0) / len(texts)
t0 = time.time(); cp.answer_logprobs(model, tok, task, ["Positive", "Negative"], save=False); t_ask = (time.time() - t0) / len(task)
t0 = time.time(); cp.generate(model, tok, task[:32], max_new_tokens=16, save=False); t_gen = (time.time() - t0) / 32
print(f"per text: probe {1000 * t_probe:.0f} ms, asking the model {1000 * t_ask:.0f} ms, "
      f"generating a 16-token answer {1000 * t_gen:.0f} ms")

**Monitoring while generating.** The probe reads every token as the model writes it, at no extra
cost: here, a story the model writes freely, and the same story pushed toward negative.

In [ ]:
trace = {}
for name, push in [("no steering", None), ("steered negative", probe.steering(-0.15))]:
    steps = []
    gen = cp.generate(model, tok, ["Tell a very short story about a student getting her exam results."], max_new_tokens=60,
                      intervention=push, probes=[probe], on_token=lambda step: steps.append(step.scores[probe.name][0]),
                      save=False)
    trace[name] = steps
    print(f"{name}: {gen[0].text!r}\n")
plt.figure(figsize=(7, 3))
for name, steps in trace.items():
    plt.plot(steps, label=name)
plt.xlabel("token"); plt.ylabel("probe score while generating"); plt.legend(); plt.show()

**Side effects of steering.** Push valence and read the interest probe of section 4 on what the
model writes: does making the text more positive also make it more "interested"? Concepts are
entangled; the random controls say how much is the push itself.

In [ ]:
side = evaluate_steering(probe, PROMPTS, doses=[-0.25, 0.25], probes=[interest])
for dose in [-0.25, 0.25]:
    for name in ["valence (eval SD)", "interested (eval SD)"]:
        e = side.row(dose)["effects"][name]
        print(f"dose {dose:+.2f} {name:22s} change {e['change']:+.2f} (random directions up to {e['null_abs_change95']:.2f})")

**On Llama 3.2 1B**: pushing valence by +0.25 also raises the interest probe by 0.33 SD (random
directions: up to 0.13).

## 11. Is the dataset any good?

**An audit, without a model.** Could anything but valence tell the poles apart? A classifier that
sees only length, or only words, cross-validated; the words that most separate the poles; and how
varied the texts are. Length should sit at chance. Words carry the concept itself ("lost",
"won't"), so a words classifier does better than chance; what to check is which words separate
the poles, and whether any of them is not the concept.

In [ ]:
audit = cp.audit_dataset(valence.train_pos, valence.train_neg, paired=True, name="valence")
for name, b in audit.baselines.items():
    print(f"{name:8s} classifier accuracy {b['accuracy']:.2f}")
print("words of the positive pole:", [w["word"] for w in audit.words["pos"][:10]])
print("words of the negative pole:", [w["word"] for w in audit.words["neg"][:10]])

**An attack.** Items written to fool a valence probe, after the dataset was final: positive texts
with negative surface features and the reverse (negation, exclamation marks, a medical topic,
...), and pairs of neutral texts that differ only in the feature. Even a clean dataset can give a
probe that follows the model's own shortcuts.

In [ ]:
attack_sets = {}
for r in attacks.records:
    a = attack_sets.setdefault(r["attack"], {"pos": [], "neg": [], "neutral": []})
    if r["kind"] == "concept":
        a[r["label"]].append(r["text"])
    else:
        a["neutral"].append((r["a"], r["b"]))
result = cp.red_team(probe, attack_sets)
nan = {"accuracy": float("nan"), "shift_mean": float("nan")}
for name, r in result.items():
    print(f"{name:22s} right side: positive {r.get('pos', nan)['accuracy']:.2f}, negative "
          f"{r.get('neg', nan)['accuracy']:.2f};  neutral pair shift {r.get('neutral', nan)['shift_mean']:+.2f} SD")

**On Llama 3.2 1B**: length at chance (0.47) and words at 0.70. The attacks find what section 5
found: negation, valence words and exclamation marks move the probe by 1.2-1.5 SD between neutral
texts that differ only in them.

## 12. Your own probe

Write a few dozen minimal pairs of your concept (the same situation with the concept on and off),
and other pairs to evaluate it; `concept_probe.guide("datasets")` says how to write them, and
what to check. Then fit it, read its controls, and attack it.

In [ ]:
MY_TRAIN_POS = ["I double-checked every number before sending the report.", "She read the contract twice before signing."]
MY_TRAIN_NEG = ["I sent the report without looking at the numbers.", "She signed the contract without reading it."]
MY_EVAL_POS = ["He tested the brakes before the long drive."]
MY_EVAL_NEG = ["He set off on the long drive without testing the brakes."]
# mine = ConceptProbe.from_texts(bundle, MY_TRAIN_POS, MY_TRAIN_NEG, MY_EVAL_POS, MY_EVAL_NEG, paired=True, name="careful")
# print(mine.metrics["d"], mine.metrics["null_d95"])
print(cp.guide("workflow")[:1500])

**A checklist**, from everything above:

- Look at what goes into the model (`preview_input`), and read later texts exactly as you trained.
- Choose the layer on items you do not report on; report d next to `null_d95` and a label-shuffle control.
- Report the number of training items, and how much the result varies across draws of them.
- Evaluate out of distribution: paraphrases, other genres, another language, attacks.
- Look for confounds before writing the data, and test for them after: length, format, keywords, negation, topic.
- Texts the model wrote under a system prompt: evaluate them without it.
- Steer against random directions of the same length, with a judge that is not the probe, and watch coherence.
- A probe that reads a concept is not proof the model uses it: ablate and patch.
- z = 0 is the midpoint between the poles, not "without the concept"; the baseline is part of the concept.
- Some concepts are not a direction: look at their geometry.